# Demo dữ liệu tổng hợp và Independent Leiden — phần Phúc

**PHUC-006-T03.** Notebook mở lại kết quả PHUC-005 đã khóa và gói T02 đã kiểm tra. Không sinh graph, chạy detector hoặc tính AMI mới; không cần tracker Đăng. Mọi bảng/hình lấy từ input có hashes, seed chính vẫn **0** đã chốt trước evaluation.

- **Node:** user giả lập. **Cạnh:** quan hệ được bộ sinh lấy mẫu, không phải dữ liệu Reddit.
- **Truth:** nhóm/sự kiện do kịch bản biết trước. **Prediction:** nhóm Leiden tìm trên graph.
- **AMI:** mức khớp hai phân hoạch; không phải tỷ lệ dự báo sự kiện đúng. **Coverage:** phần user hoạt động được gán nhóm.

## Môi trường và cách mở

Python 3.11+. Chạy ở terminal tại root repo bằng Python của môi trường dự án nếu cần cài đặt:

```powershell
& .\.venv\Scripts\python.exe -m pip install -e '.[notebook]'
```

Trong VS Code/Jupyter chọn kernel `.venv/Scripts/python.exe`, rồi **Restart Kernel → Run All**. Không cell nào cài thư viện hoặc sửa `sys.path`. Package phải được cài; input mặc định là thư mục ZIP T02 trên máy Phúc. Máy khác cần nhận/giải nén ZIP, đặt `REDDIT_SYNTHETIC_BUNDLE` tới root gói. Notebook dùng reference SHA đã version trong repo, không tự tin vào hash do input vừa nhận khai báo.

Execution T03 là smoke run trong venv hiện có; clean-install/replay nghiệm thu đầy đủ thuộc T05. Notebook trong Git không chứa output cache. Hình/CSV và bản notebook đã chạy được lưu riêng trong artifacts.

In [ ]:
import importlib.util
import sys

print("Active Python:", sys.executable)
if sys.version_info < (3, 11):
    raise RuntimeError("Chọn kernel Python 3.11+ của dự án.")
missing = [name for name in ("reddit_sna", "matplotlib", "numpy") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Thiếu {missing}. Cài package với extra [notebook] bằng Python của kernel, rồi restart.")

import json
import os
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from IPython.display import display, HTML, JSON
from IPython import get_ipython
import matplotlib.pyplot as plt
get_ipython().run_line_magic("matplotlib", "inline")
from reddit_sna.reporting.synthetic_demo import (
    load_synthetic_demo, build_synthetic_figures, export_synthetic_demo, table_html,
)

## 1. Chọn input/output và phạm vi

ROOT được tìm từ cwd hoặc `REDDIT_SNA_PROJECT`; không phụ thuộc đường dẫn cá nhân. `REDDIT_SYNTHETIC_BUNDLE` có thể trỏ tới bản giải nén cùng gói T02 ở chỗ khác. `REDDIT_SYNTHETIC_DEMO_OUTPUT` chọn output mới bên trong `artifacts/` của repo; để trống sẽ tạo tên timestamp mới. Input/output không chồng nhau, không ghi đè run cũ.

Dataset minh họa mặc định **dev-grow-easy-g11**, chọn trước khi mở điểm. Có thể đổi `REDDIT_SYNTHETIC_DEMO_DATASET` để xem dataset khác; không thay seed canonical theo score. Notebook trình bày cả hai tập của detection đã khóa; Đăng chỉ dùng evaluation sau khi tự khóa tracker/evaluator của mình.

In [ ]:
search = Path(os.environ.get("REDDIT_SNA_PROJECT", Path.cwd())).resolve()
ROOT = next((p for p in (search, *search.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/reddit_sna").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Đặt REDDIT_SNA_PROJECT tới root repo đã cài package.")

reference_path = ROOT / "document/specs/phuc/evidence/PHUC-006-T02-delivery-reference.json"
reference = json.loads(reference_path.read_text(encoding="utf-8"))
BUNDLE = Path(os.environ.get("REDDIT_SYNTHETIC_BUNDLE", ROOT / "artifacts/handoff/phuc-dang-week1-20261019-001")).resolve()
DATASET = os.environ.get("REDDIT_SYNTHETIC_DEMO_DATASET", "dev-grow-easy-g11")
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = Path(os.environ.get("REDDIT_SYNTHETIC_DEMO_OUTPUT", ROOT / "artifacts/detection-tracking" / f"phuc-006-t03-demo-{STAMP}")).resolve()
NOTEBOOK_SOURCE = ROOT / "notebooks/detection_tracking/01_synthetic_demo.ipynb"
if not OUTPUT.is_relative_to((ROOT / "artifacts").resolve()):
    raise ValueError("Demo output phải nằm trong artifacts/ của repo.")
if OUTPUT.exists() or OUTPUT.is_relative_to(BUNDLE) or BUNDLE.is_relative_to(OUTPUT):
    raise ValueError("Chọn output mới, không chồng input.")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print("Input:", BUNDLE)
print("Output mới:", OUTPUT)
print("Dataset minh họa:", DATASET)
print("Delivery manifest SHA-256:", reference["delivery_manifest_sha256"])

## 2. Kiểm tra source rồi mở dữ liệu

Helper kiểm tra bản khóa manifest của ZIP T02 và hashes/sizes toàn bộ file khai báo. Consumer T01 xác minh source runs complete, identity/reference, node/edge/truth keys và membership đủ ba seed đúng endpoint. Các metric/summary được đọc từ CSV đã hash; không tính lại hoặc thay dữ liệu để đẹp hơn.

Mỗi split có dataset IDs và generator seeds riêng. Detector seed là lần lặp trong dataset, không phải dataset độc lập. Metadata producer bên dưới là môi trường lúc sinh/chấm, khác môi trường đang vẽ notebook.

In [ ]:
start = perf_counter()
demo = load_synthetic_demo(BUNDLE, expected_delivery_sha256=reference["delivery_manifest_sha256"], dataset_id=DATASET)
print(f"Replay verification/read wall seconds (not producer runtime): {perf_counter() - start:.3f}")
source_rows = [dict(split=split, **inventory["counts"],
                    source_manifest_sha256=inventory["source_manifest_sha256"])
               for split, inventory in demo["inventories"].items()]
display(HTML(table_html(source_rows)))
print("Graph recipe đã lưu:")
display(JSON(demo["recipes"]["development"]))
print("Producer environments:")
display(JSON(demo["environments"]))

## 3. Bảng AMI/coverage và support đầy đủ

Nguồn: `runs/<split>/evaluation/detection_summary.csv`. Mỗi seed giữ riêng; mean lấy trên dataset-month có AMI hợp lệ, không trộn seeds/splits. Bảng giữ số valid/null và reason, không điền thiếu bằng 0 hoặc 1. **Coverage 1 không đồng nghĩa tìm nhóm đúng hoàn toàn.**

Policy của đồ án cho AMI null khi phân hoạch hằng, quá ít user hoặc rỗng; dissolve tháng cuối chỉ còn nhóm nền nên truth constant. Đây là trạng thái thiếu có lý do, không tự coi là lỗi pipeline. Việc phân tích sâu case khó thuộc T04.

In [ ]:
all_summary = [r for r in demo["summary_view"] if r["level"] == "all_scenarios"]
display(HTML(table_html(all_summary, ["split", "detector_seed", "seed_role", "n_rows", "n_ami_valid", "n_ami_missing", "ami_mean", "coverage_mean", "missing_reasons"])))
print("Metric rows thật:", len(demo["metrics"]), "| Summary rows thật:", len(demo["summaries"]))

## 4. Nhìn graph và nhóm tìm được

Nguồn: các paths `nodes_file`, `graph_file`, `truth_membership_file`, `detector_membership_file` trong inventory của dataset đã chọn. Vẽ **toàn bộ active nodes và cạnh** hai tháng; không lấy mẫu để tính hay thay chỉ số. Cùng user có cùng vị trí vòng tròn qua các panels, không dùng layout như bằng chứng chất lượng.

Màu/ID nhóm chỉ có nghĩa trong từng panel; không ghép nhóm truth với detector bằng tên hoặc màu. Nếu có isolate không được detector gán nhóm, dùng dấu × xám. AMI/coverage ở bảng dưới là từ full graph report.

In [ ]:
figures = build_synthetic_figures(demo)
for figure in figures.values():
    plt.close(figure)  # giữ objects để display/export, tránh Jupyter tự hiển thị trùng
display(figures["01_network"])
selected_metrics = [s["metric"] for s in demo["network"]["snapshots"]]
display(HTML(table_html(selected_metrics, ["dataset_id", "month", "detector_seed", "active_nodes", "endpoint_nodes", "assigned_nodes", "isolated_nodes", "coverage", "ami", "ami_status", "ami_reason"])))

## 5. AMI theo case và diagnostics giữa các seed

Hai hình lấy **recorded summary**, không chọn seed có điểm cao nhất. Ở hình case, số trên cột AMI là `valid / total`; mean dissolve chỉ dùng những dòng hợp lệ và vẫn báo phần thiếu. Giá trị thiếu là khoảng trống, không giả thành điểm 0. Seed 0 luôn canonical, 1/2 diagnostic; điểm khác nhau không phải ba dataset mới.

In [ ]:
display(figures["02_quality"])
display(figures["03_seeds"])

## 6. Thời gian và phạm vi đo

Nguồn: `runs/<split>/manifest.json` của lượt chạy PHUC-005 cũ. Đây là thời gian sinh/detection/evaluation **đã ghi nhận**, không phải thời gian vẽ notebook hiện tại. Stage wrapper có checkpoint/integrity overhead; detection bao gồm cả ba seeds. Tổng run có scope riêng, không cộng hoặc chia seed để tạo thời gian giả. Timing chi tiết mỗi dataset-month có sẵn trong report cũ; không nhân 36 dòng tháng thành 108 dòng thời gian seed. Peak memory chưa đo.

In [ ]:
display(HTML(table_html(demo["timings"])))
display(figures["04_timing"])
print("Peak memory: chưa đo; không có số giả để vẽ.")

## 7. Lưu đầu ra và nguồn từng hình

Output mới gồm 4 hình PNG + SVG, full metrics/summary CSV, summary view và timing CSV, `figure-sources.json` chỉ tới input paths/hashes và metadata complete ghi cuối. Helper kiểm tra lại source bytes trước/sau export. Notebook nguồn trong Git không chứa kết quả cũ; bản đã chạy và logs nằm ở artifacts evidence.

Không sửa ZIP T02 đã giao. Nếu output tồn tại, chạy lại cell cấu hình/Run All để lấy tên mới. Khi gặp lỗi, giữ output dở dang và đọc `failure.json` thay vì xem nó là complete.

In [ ]:
metadata = export_synthetic_demo(demo, figures, OUTPUT, notebook=NOTEBOOK_SOURCE)
print("Đã lưu:", OUTPUT)
display(JSON({k: metadata[k] for k in ("status", "scope", "selected_dataset", "metric_rows", "summary_rows", "figure_count", "canonical_seed", "partner_acceptance")}))
print("Nguồn figure và checksums:", OUTPUT / "figure-sources.json")

## Phần còn chờ và bước tiếp theo

Bạn đã báo nhắn Đăng; chưa có phản hồi đọc gói hoặc artifact tracker để nghiệm thu. Các bước trên chỉ trình bày detection của Phúc trên dữ liệu tổng hợp. Không có event-F1/confusion matrix, forecast accuracy, kết quả Reddit hoặc demo cặp trong notebook này.

**T04:** phân tích một case khó/null finding từ graph/truth/prediction đã lưu. **T05:** nghiệm thu replay/clean-kernel và môi trường cài đặt sạch phù hợp. **T06:** chốt gói phần Phúc và ghi riêng receipt/tracker/demo chung. Việc Đăng làm tracker không chặn các task cá nhân này.